In [83]:
import pandas as pd
import numpy as np


In [84]:
df = pd.read_csv('default of credit card clients.csv', header=1, sep=';')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

In [85]:
df = df.drop("ID", axis=1)
print(df.head())

   LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0      20000    2          2         1   24      2      2     -1     -1   
1     120000    2          2         2   26     -1      2      0      0   
2      90000    2          2         2   34      0      0      0      0   
3      50000    2          2         1   37      0      0      0      0   
4      50000    1          2         1   57     -1      0     -1      0   

   PAY_5  ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0     -2  ...          0          0          0         0       689         0   
1      0  ...       3272       3455       3261         0      1000      1000   
2      0  ...      14331      14948      15549      1518      1500      1000   
3      0  ...      28314      28959      29547      2000      2019      1200   
4      0  ...      20940      19146      19131      2000     36681     10000   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default payment next month  
0     

In [86]:
df.rename(columns={
    "PAY_0": "PAY_1",
    "default payment next month": "default"
}, inplace=True)

df.isnull().sum()

LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
default      0
dtype: int64

In [87]:
print(df.duplicated().sum())
df.drop_duplicates(inplace=True)
print(df.duplicated().sum())

35
0


In [88]:
df["default"].value_counts()
df["default"].value_counts(normalize=True)
# 1 gagal byr
# 0 byr tepat wakt0
#Lumayan imbalance 

default
0    0.778742
1    0.221258
Name: proportion, dtype: float64

In [89]:
df["EDUCATION"].value_counts()
# di informasi cuma ada 0 - 4(others) 

EDUCATION
2    14019
1    10563
3     4915
5      280
4      123
6       51
0       14
Name: count, dtype: int64

In [90]:
df["EDUCATION"] = df["EDUCATION"].replace([0,5,6], 4)
df["EDUCATION"].value_counts()

EDUCATION
2    14019
1    10563
3     4915
4      468
Name: count, dtype: int64

In [91]:
df["MARRIAGE"].value_counts()
df["MARRIAGE"] = df["MARRIAGE"].replace([0], 3)
df["MARRIAGE"].value_counts()

MARRIAGE
2    15945
1    13643
3      377
Name: count, dtype: int64

In [99]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import joblib

In [97]:
X = df.drop("default", axis=1)
y = df["default"]
scaler = StandardScaler()
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=22)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [100]:
param_grid = {
    'C': [0.01, 0.1 ,1 ,10],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [100,200,500]
}

grid = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring="accuracy")
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_
print("Best parameters: ", grid.best_params_)

Best parameters:  {'C': 1, 'max_iter': 100, 'solver': 'liblinear'}


In [104]:
y_pred = best_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("\n Classification Report")
print(classification_report(y_test, y_pred))
print("\n Confussion Matrix: ")
print(confusion_matrix(y_test, y_pred))
joblib.dump(best_model, "logistic_model.pkl")
print("Model saved to logistic_model.pkl")

Accuracy: 0.8099449357583848

 Classification Report
              precision    recall  f1-score   support

           0       0.82      0.98      0.89      4650
           1       0.74      0.24      0.36      1343

    accuracy                           0.81      5993
   macro avg       0.78      0.61      0.62      5993
weighted avg       0.80      0.81      0.77      5993


 Confussion Matrix: 
[[4537  113]
 [1026  317]]
Model saved to logistic_model.pkl
